In [ ]:
!pip install -Uqq fastbook
!pip install --upgrade ipywidgets jupyterlab_widgets
!pip install voila
!jupyter serverextension enable --sys-prefix voila 

import io
from PIL import Image as PILImage
import ipywidgets as widgets
from fastai.vision.widgets import *
from fastai.vision.all import *
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import io
from IPython.display import display, clear_output


for dirname, _, filenames in os.walk('dataset/bears'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: jupyterlab_widgets
    Found existing installation: jupyterlab_widgets 1.1.11
    Uninstalling jupyterlab_widgets-1.1.11:
      Successfully uninstalled jupyterlab_widgets-1.1.11
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.8.5
    Uninstalling ipywidgets-7.8.5:
      Successfully uninstalled ipywidgets-7.8.5
   ━━━━━━━━━━━━━━━━━━━

In [ ]:
path = Path('dataset/bears')
fns = get_image_files(path)

bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label, # This is the magic line
    item_tfms=Resize(128)
)

dls = bears.dataloaders(path)

bears = bears.new(
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms())
dls = bears.dataloaders(path)

In [9]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(4)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/binhkhuu/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:07<00:00, 6.69MB/s]


epoch,train_loss,valid_loss,error_rate,time
0,1.449565,0.274674,0.105263,00:12


epoch,train_loss,valid_loss,error_rate,time
0,0.255378,0.138480,0.035088,00:06
1,0.176463,0.107120,0.035088,00:05
2,0.146790,0.167338,0.035088,00:05
3,0.127582,0.159520,0.035088,00:05


In [10]:
learn.export()

# Check that the file exist
path = Path()
path.ls(file_exts='.pkl')

[Path('export.pkl')]

In [11]:
learn_inf = load_learner(path/'export.pkl')

/Users/binhkhuu/Code/AI/FastAI/Course/Ch2/bearsbook/.venv/lib/python3.14/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


In [ ]:
# --- ASSUMING YOU ALREADY HAVE YOUR MODEL LOADED ---
# learn_inf = load_learner(...) 

# 1. Create the UI elements (The "View")
btn_upload = widgets.FileUpload(accept='image/*', multiple=False)
output_area = widgets.Output() # This is a special widget to hold our results

# 2. Define the Handler (The "Controller" - The logic that runs on click)
def handle_upload(change):
    """
    This function is triggered automatically when 'btn_upload.value' changes.
    """
    # Clear the previous output so we don't see old predictions/images
    with output_area:
        clear_output() 
        
        # Check if there is actually a file in the widget
        if not btn_upload.value:
            print("No file uploaded yet.")
            return

        try:
            print("Processing new upload...")
            
            # A. Extract bytes from the widget
            uploaded_file = btn_upload.value[0]
            image_bytes = uploaded_file['content']
            
            # B. Convert bytes to a PIL Image (for display)
            img = PILImage.open(io.BytesIO(image_bytes))
            
            # C. Save to disk (as you did in your snippet) for the model to read
            temp_path = "/kaggle/working/temp_image.jpg"
            with open(temp_path, "wb") as f:
                f.write(image_bytes)
            
            # D. Run Inference
            pred, pred_idx, probs = learn_inf.predict(temp_path)
            
            # E. Display the results to the user
            print("-" * 30)
            print(f"PREDICTION: {pred}")
            print(f"CONFIDENCE: {probs[pred_idx]:.2%}")
            print("-" * 30)
            display(img)
            
        except Exception as e:
            print(f"An error occurred during prediction: {e}")

# 3. The "Magic" Link (The Observer)
# This tells the button: "Watch your 'value' property. If it changes, run handle_upload."
btn_upload.observe(handle_upload, names='value')

# 4. Display everything to the user
print("AI Image Classifier App")
print("Click the button below to upload an image:")
display(btn_upload)
display(output_area)


AI Image Classifier App
Click the button below to upload an image:


FileUpload(value=(), accept='image/*', description='Upload')

Output()